# 🧪 Praktikum Alternatif Bab 03: Penanganan Missing Values & Extreme Imbalance pada Transaksi Kartu Kredit
### Universitas Indraprasta PGRI (Unindra) - Laboratorium Informatika
**Mata Kuliah**: Machine Learning  
**Modul Referensi**: Bab 03  
**Fokus Topik**: Pembersihan Data, KNNImputer, dan SMOTE (Studi Kasus: Deteksi Fraud Finansial)  

---

## 📌 Perbedaan dengan Modul Utama:
> Berbeda dari modul utama (Missing data properti). Di sektor perbankan, fraud hanya terjadi < 2% dari seluruh transaksi. Kita menggunakan KNNImputer untuk nilai hilang dan SMOTE untuk menyeimbangkan kelas tanpa menduplikasi data mentah.

---

## 🎯 Tujuan Praktikum:
1. Memahami penerapan algoritma pada studi kasus riil yang kontekstual dan mutakhir.
2. Melatih pembersihan, eksplorasi, transformasi, hingga evaluasi metrik kinerja model.
3. Menumbuhkan daya nalar komputasi mahasiswa di luar dataset standar modul kuliah.


## 1. Persiapan Environment & Import Pustaka

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


## 2. Inisialisasi & Karakterisasi Dataset Alternatif

In [ ]:
# Pembuatan Dataset Simulasi Transaksi Keuangan
np.random.seed(101)
n_tx = 600

amount = np.random.exponential(scale=150, size=n_tx)
dist_from_home = np.random.gamma(shape=2, scale=10, size=n_tx)
dist_from_last_tx = np.random.exponential(scale=5, size=n_tx)
ratio_to_median = np.random.normal(1.0, 0.4, size=n_tx)

# Label Fraud (~3%)
is_fraud = ((amount > 600) & (dist_from_last_tx > 20) | (ratio_to_median > 3.0)).astype(int)

df_fraud = pd.DataFrame({
    'amount': np.round(amount, 2),
    'dist_home_km': np.round(dist_from_home, 1),
    'dist_last_tx_km': np.round(dist_from_last_tx, 1),
    'ratio_to_median': np.round(ratio_to_median, 2),
    'is_fraud': is_fraud
})

# Injeksi Missing Values secara acak (MCAR)
mask_nan = np.random.rand(n_tx) < 0.10
df_fraud.loc[mask_nan, 'dist_home_km'] = np.nan

df_fraud.to_csv('../../data/raw/alternatif/financial_fraud_data.csv', index=False)
print("Jumlah Missing Values per Kolom:\n", df_fraud.isnull().sum())
print("Distribusi Target Awal:", Counter(df_fraud['is_fraud']))


## 3. Implementasi Algoritma & Analisis Kinerja Model

In [ ]:
# 1. Imputasi dengan KNNImputer
imputer = KNNImputer(n_neighbors=3)
X_imputed = imputer.fit_transform(df_fraud.drop('is_fraud', axis=1))
y = df_fraud['is_fraud'].values

# 2. Resampling dengan SMOTE (Synthetic Minority Over-sampling Technique)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_imputed, y)

print("Distribusi Target Setelah SMOTE:", Counter(y_resampled))

# 3. Evaluasi Model
rf = RandomForestClassifier(random_state=42, n_estimators=50)
rf.fit(X_resampled, y_resampled)
y_pred = rf.predict(X_imputed)
print("\nHasil Klasifikasi pada Data Asli Imputed:")
print(classification_report(y, y_pred))


## 4. Tugas Mandiri & Pertanyaan Diskusi Laboratorium:
1. **Analisis Sensitivitas**: Bagaimana perubahan hyperparameter mempengaruhi metrik evaluasi pada dataset ini?
2. **Kesesuaian Algoritma**: Mengapa algoritma yang digunakan cocok untuk karakteristik data di studi kasus ini?
3. **Pertimbangan Bisnis/Etika**: Jika model ini diimplementasikan di dunia nyata, apa risiko kesalahan prediksi (False Positive vs False Negative) yang paling berbahaya?
